# Clone Repository and set up the Environment

In [29]:
!pwd

/content/robust-eeg-models


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

fatal: destination path 'robust-eeg-models' already exists and is not an empty directory.


In [3]:
%cd robust-eeg-models

/content/robust-eeg-models


In [5]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [103]:
!git add .
!git commit -m "EEGMamanba model solve"
!git push origin main

[main f480bad] EEGMamanba model solve
 1 file changed, 229 insertions(+)
 create mode 100644 models/eeg_mamba_fft.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 8 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.60 KiB | 2.60 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/VictoryChianumba/robust-eeg-models.git
   dedd04a..f480bad  main -> main


In [7]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna

# Clean uninstall
!pip uninstall -y braindecode

# Install latest code from GitHub (which includes CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 60.4 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 137.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 165.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 48.7 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 76.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 130.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 119.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 71.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 78.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 141.0 MB/s  0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing ins

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]
  Cloning https://github.com/braindecode/braindecode.git (to revision master) to /tmp/pip-req-build-zo8frmnt
  Running command git clone --filter=blob:none --quiet https://github.com/braindecode/braindecode.git /tmp/pip-req-build-zo8frmnt
  Resolved https://github.com/braindecode/braindecode.git to commit 1c3fb2d196d49734772f3959337070f885915ce4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 285.5 MB/s  0:00:00
  Created wheel for braindecode: filename=braindecode-1.1.0-py3-none-any.whl size=287253 sha256=fde8cdca00331fca7f8fbf4091e5cb36535f8aa8990dc62345db9784d3093da1
  Stored in directory: /tmp/pip-ephem-wheel-cache-e3w89sw8/wheels/b6/8b/2b/0da876924d16f36b5bdd43797902e55d90426ed71fd51642f2
Successfully built braindecode
  Attempting uninstall: pandas
    Found existing installation: p

In [7]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.0.0
CTNet is available ✅


In [8]:
import torch
import importlib
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

import numpy as np
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


In [9]:
from braindecode.models import EEGNetv4, CTNet
from braindecode.datasets import MOABBDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split


# Loading data for training

In [10]:
dataset = MOABBDataset(dataset_name="BNCI2014001", subject_ids=[1])

/usr/local/lib/python3.11/dist-packages/moabb/datasets/download.py:56: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


MNE_DATA is not already configured. It will be set to default location in the home directory - /root/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 42.8M/42.8M [00:00<00:00, 71.0GB/s]
SHA256 hash of downloaded file: 054f02e70cf9c4ada1517e9b9864f45407939c1062c6793516585c6f511d0325
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 43.8M/43.8M [

In [91]:
%%writefile models/eeg_mamba_fft.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from braindecode.models.base import EEGModuleMixin

# ------------------------------------------------------------------
# 1. FFT-Based Mamba Implementation (No external dependencies)
# ------------------------------------------------------------------
class FFTMamba(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, bidirectional: bool = True,
                 dropout: float = 0.3):
        super().__init__()
        self.d_state = d_state
        self.bidir   = bidirectional
        self.A_log   = nn.Parameter(torch.randn(d_state) * 0.02)
        self.B_proj  = nn.Linear(d_model, d_state)
        self.C_proj  = nn.Linear(d_state, d_model)
        self.D       = nn.Parameter(torch.ones(d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        A = -torch.exp(self.A_log).unsqueeze(0)
        u = self.B_proj(x)

        n_fft = 2 ** (T - 1).bit_length()
        t = torch.arange(T, dtype=torch.float, device=x.device)
        K = torch.exp(A * t.unsqueeze(1)).to(torch.float32)
        if self.bidir:
            K = K + torch.flip(K, dims=[0])

        K_f = fft.rfft(K, n=n_fft, dim=0)
        u_f = fft.rfft(u, n=n_fft, dim=1)
        y_f = K_f.unsqueeze(0) * u_f
        y = fft.irfft(y_f, n=n_fft, dim=1)[..., :T, :]

        out = self.dropout(self.C_proj(y)) + self.D * x
        return out

class STAdaptive(nn.Module):
    """Spatially-adaptive 1×1 conv + class token."""
    def __init__(self, n_chans: int, d_model: int = 128):
        super().__init__()
        self.proj = nn.Conv1d(n_chans, d_model, kernel_size=1, bias=False)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, x):               # (B, C, T)
        z = self.proj(x).transpose(1, 2)        # (B, T, D)
        cls = self.cls.expand(z.size(0), -1, -1)
        return torch.cat([cls, z], dim=1)       # (B, T+1, D)


class BiMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, dropout: float = 0.1, ffn_mult: int = 2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.mamba = FFTMamba(d_model, d_state=d_state, bidirectional=True, dropout=dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_mult * d_model, d_model),
        )

    def forward(self, x):
        x = x + self.mamba(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class TaskMoE(nn.Module):
    """Task-aware Mixture-of-Experts head."""
    def __init__(self, d_model: int, n_classes: int,
                 n_experts: int = 9, k: int = 2):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Linear(d_model, n_classes) for _ in range(n_experts)
        ])
        self.gate = nn.Linear(d_model, n_experts)
        self.k = k

    def forward(self, x):
      logits = self.gate(x)
      probs = F.softmax(logits, dim=-1)
      topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)

      # Compute all expert outputs
      expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)  # (B, n_experts, n_classes)

      # Select top-k experts using advanced indexing
      batch_indices = torch.arange(x.size(0), device=x.device).unsqueeze(1)  # (B, 1)
      selected_outputs = expert_outputs[batch_indices, topk_idx]  # (B, k, n_classes)

      # Weight and sum
      weights = topk_vals.unsqueeze(-1)  # (B, k, 1)
      y = (weights * selected_outputs).sum(dim=1)  # (B, n_classes)

      return y
# ------------------------------------------------------------------
# 2. Fixed Braindecode Wrapper
# ------------------------------------------------------------------
class EEGMamba(EEGModuleMixin, nn.Module):
    """Braindecode-compatible EEGMamba model."""

    def __init__(
        self,
        # Braindecode standard parameters
        n_chans=None,
        n_outputs=None,
        n_times=None,
        chs_info=None,
        input_window_seconds=None,
        sfreq=None,
        # EEGMamba specific parameters
        d_model=128,
        n_layers=8,
        n_experts=9,
        k=2,
        # Backward compatibility aliases
        in_chans=None,
        n_classes=None,
        input_window_samples=None,
    ):

        # Initialize base class
        super().__init__(
            n_outputs=n_outputs,
            n_chans=n_chans,
            chs_info=chs_info,
            n_times=n_times,
            input_window_seconds=input_window_seconds,
            sfreq=sfreq,
        )

        # Store model parameters
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_experts = n_experts
        self.k = k

        # Build the model
        self.st_adaptive = STAdaptive(self.n_chans, d_model)
        self.layers = nn.ModuleList([
            BiMambaBlock(d_model) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        # Simple classification head (no MoE for standard compatibility)
        self.classifier = nn.Linear(d_model, self.n_outputs)

        # Optional: MoE head (use when you want advanced features)
        self.moe_head = TaskMoE(d_model, self.n_outputs, n_experts, k)
        self.use_moe = False  # Toggle this for MoE vs standard mode

        # Initialize weights
        self._initialize_weights()

    def forward(self, x):
        """
        Forward pass compatible with Braindecode.

        Args:
            x: Input tensor [batch_size, n_chans, n_times]

        Returns:
            logits: Output tensor [batch_size, n_outputs]
        """
        # Spatial-temporal adaptive processing
        x = self.st_adaptive(x)  # (B, T+1, D)

        # Bidirectional Mamba layers
        for layer in self.layers:
            x = layer(x)

        # Use class token
        x = self.norm(x).mean(dim=1)   # (B, D)

        # Classification
        if self.use_moe:
            return self.moe_head(x)  # Only expects tensor
        else:
            return self.classifier(x)  # Only returns tensor

    def get_output_shape(self):
        """Required method for Braindecode compatibility."""
        with torch.no_grad():
            dummy_input = torch.zeros(1, self.n_chans, self.n_times)
            output = self.forward(dummy_input)
            return output.shape

    def enable_moe(self, enable=True):
        """Enable/disable MoE head."""
        self.use_moe = enable

    def _initialize_weights(self):
        """Initialize model weights."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

# ------------------------------------------------------------------
# 3. Factory Function for Easy Usage
# ------------------------------------------------------------------
def create_eegmamba(n_chans, n_outputs, n_times, **kwargs):
    """
    Factory function to create EEGMamba model.

    Args:
        n_chans: Number of EEG channels
        n_outputs: Number of classes
        n_times: Number of time samples
        **kwargs: Additional model parameters

    Returns:
        EEGMamba model instance
    """
    return EEGMamba(
        n_chans=n_chans,
        n_outputs=n_outputs,
        n_times=n_times,
        **kwargs
    )


Overwriting models/eeg_mamba_fft.py


In [92]:
# If you've already imported the module
import importlib
import models.eeg_mamba_fft
importlib.reload(models.eeg_mamba_fft)

# Then re-import what you need
from models.eeg_mamba_fft import EEGMamba, create_eegmamba

In [13]:
from braindecode.preprocessing import Preprocessor,exponential_moving_standardize, preprocess


low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

/usr/local/lib/python3.11/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


In [14]:
from braindecode.preprocessing import create_windows_from_events

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
)


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']


In [15]:
splitted = windows_dataset.split("session")
train_set = splitted["0train"]  # Session train
test_set = splitted["1test"]  # Session evaluation

# Add data Augmentation to before training

In [93]:
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from models.eeg_mamba_fft import create_eegmamba, EEGMamba
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

n_classes = len(torch.unique(torch.tensor([sample[1] for sample in windows_dataset])))
classes = list(range(n_classes))
# Extract number of chans and time steps from dataset
n_channels = windows_dataset[0][0].shape[0]
n_times = windows_dataset[0][0].shape[1]

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    n_layers = 2,
    n_experts = 4

)

model.use_moe = True

# Display torchinfo table describing the model
print(model)

# Send model to GPU
if cuda:
    model.cuda()

Layer (type (var_name):depth-idx)             Input Shape               Output Shape              Param #                   Kernel Shape
EEGMamba (EEGMamba)                           [1, 22, 1125]             [1, 4]                    516                       --
├─STAdaptive (st_adaptive): 1-1               [1, 22, 1125]             [1, 1126, 128]            128                       --
│    └─Conv1d (proj): 2-1                     [1, 22, 1125]             [1, 128, 1125]            2,816                     [1]
├─ModuleList (layers): 1-2                    --                        --                        --                        --
│    └─BiMambaBlock (0): 2-2                  [1, 1126, 128]            [1, 1126, 128]            --                        --
│    │    └─LayerNorm (norm1): 3-1            [1, 1126, 128]            [1, 1126, 128]            256                       --
│    │    └─FFTMamba (mamba): 3-2             [1, 1126, 128]            [1, 1126, 128]            4,

/usr/local/lib/python3.11/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


In [57]:
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset, predefined_split
from torch.utils.data import Subset

X_train = SliceDataset(train_set, idx=0)
y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
train_indices, val_indices = train_test_split(
    X_train.indices_, test_size=0.2, shuffle=False
)
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)

In [100]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score
from skorch.callbacks import LRScheduler, EarlyStopping
from braindecode.models import CTNet

from braindecode import EEGClassifier

from braindecode.training import mixup_criterion

from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader

from braindecode.augmentation import Compose, SmoothTimeMask, Mixup

transforms = [
    FrequencyShift(probability=0.5, sfreq=250, max_delta_freq=1.0),
    GaussianNoise(probability=0.5, std=0.1),
    # SmoothTimeMask(probability = 0.5, mask_len_samples=25),
    Mixup(alpha=0.2)
]


lr = 0.001
weight_decay = 0.0001
batch_size = 64
n_epochs = 200

train_val_split = KFold(n_splits=5, shuffle=False)

clf = EEGClassifier(
    model,
    iterator_train=AugmentedDataLoader,
    iterator_train__transforms=transforms,
    criterion=torch.nn.CrossEntropyLoss,
    train_split=predefined_split(val_subset),
    optimizer=torch.optim.AdamW,
    optimizer__lr=lr,
    optimizer__weight_decay=weight_decay,
    batch_size=batch_size,
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

In [ ]:




param_grid_eegnet = {
    "optimizer__lr": [0.001, 0.000625],
    "optimizer__weight_decay": [0.005, 0.00001],
    "batch_size": [64, 128],
}
param_grid_moe = {
    "optimizer__lr": [0.0016, 0.0032, 0.0064],
    "optimizer__weight_decay": [5e-5, 1e-4, 5e-4],
    "batch_size": [64, 128],
}
param_grid = param_grid_moe

# By setting n_jobs=-1, grid search is performed
# with all the processors, in this case the output of the training
# process is not printed sequentially
search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=train_val_split,
    return_train_score=True,
    scoring="accuracy",
    refit=True,
    verbose=1,
    error_score="raise",
    n_jobs=1,
)

search.fit(X_train, y_train)
search_results = pd.DataFrame(search.cv_results_)

best_run = search_results[search_results["rank_test_score"] == 1].squeeze()

best_parameters = best_run["params"]

In [20]:
best_params = {'batch_size': 128, 'optimizer__lr': 0.0016, 'optimizer__weight_decay': 0.0005}

In [46]:
print(best_params)

{'batch_size': 128, 'optimizer__lr': 0.0016, 'optimizer__weight_decay': 0.0005}


In [101]:
from braindecode.augmentation import Mixup, AugmentedDataLoader
from braindecode.training import mixup_criterion

# 1.  build transforms list
transforms = [
    FrequencyShift(probability=0.5, sfreq=250, max_delta_freq=1.0),
    GaussianNoise(probability=0.5, std=0.03),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]

# # --- freeze FFT backbone ---
# for p in model.st_adaptive.parameters():
#     p.requires_grad = False

# for blk in model.layers:
#     for p in blk.parameters():  # FIX: Properly iterate through parameters
#         p.requires_grad = False

# for p in model.norm.parameters():
#     p.requires_grad = False

# --- unfreeze expert heads & gate ---

# Enable MoE head instead of standard classifier
model.enable_moe(False)

# for p in model.moe_head.parameters():
#     p.requires_grad = True

class MixupLoss(nn.Module):
    def __init__(self, base_criterion=None):
        super().__init__()
        # IMPORTANT: reduction='none' so we can weight per-sample
        self.base = base_criterion or nn.CrossEntropyLoss(reduction='none')

    def forward(self, preds, target):
        y1, y2, lam = target  # (B,), (B,), (B,) or scalar
        # Compute per-sample CE
        loss1 = self.base(preds, y1)          # shape: [B]
        loss2 = self.base(preds, y2)          # shape: [B]
        if not torch.is_tensor(lam):
            lam = torch.tensor(lam, device=preds.device, dtype=loss1.dtype)
        lam = lam.to(loss1.dtype)
        if lam.dim() == 0:
            # scalar lambda (rare here) -> just convex-combine scalars then mean
            return (lam * loss1 + (1 - lam) * loss2).mean()
        # per-sample lambda
        return (lam * loss1 + (1 - lam) * loss2).mean()


# Create new classifier with best parameters
final_clf = EEGClassifier(
    model,

    iterator_train=AugmentedDataLoader,
    iterator_train__transforms=transforms,
    iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    optimizer=torch.optim.AdamW,
    train_split=predefined_split(test_set),  # Use all training data
    optimizer__lr=best_params['optimizer__lr'],
    optimizer__weight_decay=best_params['optimizer__weight_decay'],
    batch_size=best_params['batch_size'],
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=20, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

# Train on full training set
final_clf.fit(train_set, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = final_clf.score(test_set, y=y_test)
print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss      lr     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------  ------
      1            0.5556        2.9319       0.5382            0.5382        0.9409  0.0016  0.7332
      2            0.5104        1.6312       0.4132            0.4132        2.2864  0.0016  0.6698
      3            0.5035        1.8827       0.5556            0.5556        1.8258  0.0016  0.6731
      4            0.6111        1.7061       0.4792            0.4792        1.3879  0.0016  0.6889
      5            0.7014        1.1008       0.6042            0.6042        0.8974  0.0016  0.6558
      6            0.4062        0.8180       0.4653            0.4653        1.0886  0.0016  0.6840
      7            0.4653        1.2029       0.5000            0.5000        0.9954  0.0016  0.6595
      8            0.6354        0.9520       0.5521            0.5521        0.9704  0.001

In [43]:
from braindecode import EEGClassifier

# Extract best parameters from GridSearch
# best_params = search.best_params_

# Create new classifier with best parameters
final_clf = EEGClassifier(
    model,
    iterator_train=AugmentedDataLoader,
    iterator_train__transforms=transform,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    train_split=None,  # Use all training data
    optimizer__lr=best_params['optimizer__lr'],  # Use best lr other parameters from best_params
    optimizer__weight_decay=best_params['optimizer__weight_decay'],
    batch_size=best_params['batch_size'],  # Use best batch size
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

# Train on full training set
final_clf.fit(train_set, y=None)

# evaluated the model after training
y_test = test_set.get_metadata().target
test_acc = final_clf.score(test_set, y=y_test)
print(f"Test acc: {(test_acc * 100):.2f}%")

AttributeError: 'tuple' object has no attribute 'type'

In [ ]:
train_val_split = KFold(n_splits=5, shuffle=False)
# By setting n_jobs=-1, cross-validation is performed
# with all the processors, in this case the output of the training
# process is not printed sequentially
cv_results = cross_val_score(
    clf, X_train, y_train, scoring="accuracy", cv=train_val_split, n_jobs=1
)
print(
    f"Validation accuracy: {np.mean(cv_results * 100):.2f}"
    f"+-{np.std(cv_results * 100):.2f}%"
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

def plot_k_fold(ax, cv, all_dataset, X_train, y_train, test_set):
    """Create a sample plot for training, validation, testing."""
    bd_cmap = [
        "#3A6190",
        "#683E00",
        "#2196F3",
        "#DDF2FF",
    ]

    ax.barh("Original\nDataset", len(all_dataset), left=0, height=0.5, color=bd_cmap[0])

    # Generate the training/validation/testing data fraction visualizations for each CV split
    for ii, (tr_idx, val_idx) in enumerate(cv.split(X=X_train, y=y_train)):
        n_train, n_val, n_test = len(tr_idx), len(val_idx), len(test_set)
        n_train2 = n_train + n_val - max(val_idx) - 1
        ax.barh("cv" + str(ii + 1), min(val_idx), left=0, height=0.5, color=bd_cmap[1])
        ax.barh(
            "cv" + str(ii + 1), n_val, left=min(val_idx), height=0.5, color=bd_cmap[2]
        )
        ax.barh(
            "cv" + str(ii + 1),
            n_train2,
            left=max(val_idx) + 1,
            height=0.5,
            color=bd_cmap[1],
        )
        ax.barh(
            "cv" + str(ii + 1),
            n_test,
            left=n_train + n_val,
            height=0.5,
            color=bd_cmap[3],
        )

    ax.invert_yaxis()
    ax.set_xlim([-int(0.1 * len(all_dataset)), int(1.1 * len(all_dataset))])
    ax.set(xlabel="Number of samples.", title="KFold Train-Test-Valid split")
    ax.legend(
        [Patch(color=bd_cmap[i]) for i in range(4)],
        ["Original set", "Training set", "Validation set", "Testing set"],
        loc="lower center",
        ncols=2,
    )
    ax.text(
        -0.07,
        0.45,
        "Train-Valid-Test split",
        rotation=90,
        verticalalignment="center",
        horizontalalignment="left",
        transform=ax.transAxes,
    )
    return ax


fig, ax = plt.subplots(figsize=(15, 7))
plot_k_fold(
    ax,
    cv=train_val_split,
    all_dataset=windows_dataset,
    X_train=X_train,
    y_train=y_train,
    test_set=test_set,
)

First, let's remove the existing cloned repository and change back to the root directory.

After running these cells, please try running the cell that imports `CTNet` again (cell `nsMZ3JXEjfYP`).